In [1]:
import warnings
from causalnex.structure import StructureModel
warnings.filterwarnings("ignore")  # silence warnings
from causalnex.structure.notears import from_pandas
from causalnex.plots import plot_structure, NODE_STYLE, EDGE_STYLE
from causalnex.network import BayesianNetwork
import pandas as pd
from causalnex.discretiser import Discretiser
import numpy as np
from sklearn.model_selection import train_test_split
from causalnex.structure.notears import from_pandas_lasso
from IPython.display import Image
from causalnex.evaluation import classification_report
from causalnex.evaluation import roc_auc
from causalnex.inference import InferenceEngine
from causalnex.evaluation import classification_report
import copy
import networkx as nx
import turtorial_utils as utils

In [5]:
df_lt = pd.read_csv("/Users/vladasverkelis/Documents/Doktorantūra/Straipsnis_1/Context_influance_to_EU_structural_funds/Data/y/lt_2016_2022_bn.csv")

In [7]:
# df_lt = df_lt.drop(["Unnamed: 0"], axis = 1)

In [8]:
df_lt

,GDPC,GDPCG,VABI,EMP,PAG,QOA,QOR,YUA,GGAG,REST,...,LBGD,PC,ARP,ROL,GE,COC,RQ,SSB,PT,EUPC
0,13491.939909,3.877441,26.00,75.2,-6.000000,4.4,4.9,14.45,25.000000,25.612,...,-0.460837,-12.7,21.9,1.00,1.03,0.68,1.12,92.931332,85.480409,115.573864
1,14871.586227,5.888726,25.72,76.0,-14.000000,4.4,4.7,13.27,-79.000000,26.038,...,0.165584,-11.6,22.9,0.96,0.93,0.52,1.15,92.984744,85.497006,152.993646
2,16298.029583,5.790665,25.58,77.8,0.000000,4.6,4.7,11.12,86.000000,24.695,...,0.413734,-5.0,22.9,0.92,1.03,0.47,1.09,93.053500,87.887522,287.075911
3,17516.070462,4.979384,25.26,78.2,9.000000,4.9,4.8,11.85,128.000000,25.474,...,0.441721,-0.8,20.6,0.99,1.01,0.67,1.15,93.175686,88.744373,188.356791
4,17885.411844,0.068446,24.79,76.7,5.000000,4.4,4.7,19.56,5.000000,26.773,...,-6.374068,0.3,20.9,0.95,1.01,0.78,1.08,93.273709,88.915657,514.521647
5,20182.333587,6.455358,24.86,77.4,-14.000000,4.4,4.7,14.30,-275.142857,28.167,...,-1.419204,-1.7,20.0,1.07,1.02,0.82,1.27,93.083794,87.304994,405.630623
6,23822.056413,1.695286,25.64,79.0,1.066667,4.4,4.7,11.87,-275.142857,29.599,...,-0.897036,18.1,20.9,1.06,0.99,0.75,1.30,93.083794,87.304994,331.287996


In [21]:
lt = pd.DataFrame()
lt = df_lt.copy()
lt['EUPC_1'] = lt['EUPC'].shift(1)
lt['GDPCG_3'] = lt['GDPCG'].shift(3)
lt['EUPC_3'] = lt['EIAP'].shift(3)
lt['FDI_3'] = lt['FDI'].shift(3)

In [6]:
lt = lt.dropna()

In [7]:
sm = StructureModel()

In [8]:
sm.add_edges_from([
    ('FDI_3', 'EUPC'),
])

In [9]:
sm.edges

OutEdgeView([('FDI_3', 'EUPC')])

In [10]:
viz = plot_structure(
    sm,
    all_node_attributes=NODE_STYLE.WEAK,
    all_edge_attributes=EDGE_STYLE.WEAK,
)

viz.toggle_physics(False)
viz.show("Graphs/fully_connected_lt_2016_2022.html")

Graphs/fully_connected_lt_2016_2022.html


In [11]:
bn = BayesianNetwork(sm)

In [12]:
discretised_lt = pd.DataFrame(index=lt.index)

for col in lt.columns:
    no_unique = lt[col].nunique()
    
    if no_unique <= 1:
        discretised_lt[col] = 0
    else:
        try:
            discretised_lt[col] = pd.qcut(
                lt[col], 
                q=min(2, no_unique),  
                labels=False,
                duplicates='drop'
            ).astype(int)
        except ValueError:
            discretised_lt[col] = lt[col].rank(method='dense').astype(int) - 1

print("Discretised data:")
print(discretised_lt)

Discretised data:
   FDI  QOR  QOA  EIAP  PDPS  PGP  VABI  GCFP  GGFC  GDPCG  ...  PR  RQ  COC  \
3    2    2    0     1     2    0     0     0     2      0  ...   0   0    0   
4    2    2    0     0     1    0     2     2     2      2  ...   0   0    0   
5    0    1    2     1     1    1     2     0     1      2  ...   0   1    1   
6    1    1    2     0     0    1     1     1     0      1  ...   0   1    1   
7    0    0    1     2     0    2     1     1     0      1  ...   0   2    2   
8    1    0    1     2     0    2     0     2     1      0  ...   0   2    2   

   ROL  CPI  EUPC  PDPS_1  GDPC_1  EIAP_3  FDI_3  
3    0    0     0       2       0       2      2  
4    0    0     1       2       0       0      1  
5    1    1     1       1       1       2      0  
6    1    1     2       1       1       1      1  
7    2    2     2       0       2       0      2  
8    2    2     0       0       2       1      0  

[6 rows x 31 columns]


In [13]:
discretised_lt = discretised_lt.reset_index(drop=True)
bn.fit_node_states(discretised_lt)
baseline_auc = utils.get_avg_auc_all_info(discretised_lt, bn)
print(f"Baseline AUC: {baseline_auc}")

Processing fold 0 using 7 cores takes 3.0212411880493164 seconds
Processing fold 1 using 7 cores takes 2.8907082080841064 seconds
Processing fold 2 using 7 cores takes 2.8917040824890137 seconds
Processing fold 3 using 7 cores takes 2.888110876083374 seconds
Processing fold 4 using 7 cores takes 2.8721139430999756 seconds
Baseline AUC: 0.14375


In [14]:
edges_to_add = [('LV', 'FDI_3'), ('LV', 'EUPC')]
edges_to_remove = [('FDI_3', 'EUPC')]

bn_with_lv = copy.deepcopy(bn)
bn_with_lv.add_node(
    "LV",
    edges_to_add=edges_to_add,
    edges_to_remove=edges_to_remove,
)

In [15]:
viz = utils.plot_pretty_structure(bn_with_lv.structure, edges_to_highlight=edges_to_add)
viz.show("Graphs/node_added_lt_2016_2022.html")

Graphs/node_added_lt_2016_2022.html


In [16]:
discretised_lt['LV'] = None
lv_states = [0, 1, 2, 3, 4]

proposed_auc = utils.get_avg_auc_lvs(discretised_lt, bn_with_lv, lv_states)
print(f"AUC from adding LV between 'FDI_3' and 'EUPC': {proposed_auc}")

Processing fold 0 using 7 cores takes 3.1679699420928955 seconds
Processing fold 1 using 7 cores takes 3.256304979324341 seconds
Processing fold 2 using 7 cores takes 3.081022024154663 seconds
Processing fold 3 using 7 cores takes 3.1393959522247314 seconds
Processing fold 4 using 7 cores takes 3.1722989082336426 seconds
AUC from adding LV between 'FDI_3' and 'EUPC': 0.025
